In [7]:
# OCR + Fuzzy Match Sample Pipeline for Cosmetic Ingredient Detection

import pytesseract
pytesseract.pytesseract.tesseract_cmd = '/opt/homebrew/bin/tesseract'

from PIL import Image
import re
from rapidfuzz import fuzz, process

In [8]:
# Sample database for testing
ingredient_db = {
    "aqua": {"risk": "Safe", "description": "Water, used as a solvent"},
    "glycerin": {"risk": "Safe", "description": "Humectant that hydrates skin"},
    "niacinamide": {"risk": "Safe", "description": "Vitamin B3, brightens skin"},
    "butyrospermum parkii butter": {"risk": "Safe", "description": "Shea Butter, moisturizes"},
    "dimethicone": {"risk": "Low", "description": "Silicone used for smoothing"},
    # Add more ingredients as needed...
}
ingredient_db = {k.lower(): v for k, v in ingredient_db.items()}  # Normalize keys


In [9]:

# Step 1: OCR function
def extract_text_from_image(image_path):
    image = Image.open(image_path)
    return pytesseract.image_to_string(image)



In [10]:

# Step 2: Clean and parse ingredients
def clean_and_split_ingredients(text):
    text = text.lower().replace("ingredients:", "").strip()
    text = re.sub(r'\s*\(and\)\s*', ',', text)
    text = re.sub(r'\band\b', ',', text)
    text = text.replace('\n', ',')
    items = [i.strip(" .-") for i in text.split(',') if i.strip()]
    return items



In [11]:
def match_ingredients(ingredient_list):
    results = []
    for item in ingredient_list:
        result = process.extractOne(item.lower(), ingredient_db.keys(), scorer=fuzz.token_sort_ratio)
        if result:
            match, score = result[0], result[1]
            if score > 80:
                data = ingredient_db[match]
                results.append({"input": item, "match": match, "risk": data['risk'], "desc": data['description']})
                continue
        results.append({"input": item, "match": None, "risk": "Unknown", "desc": "Ingredient not found in database."})
    return results


In [13]:
# Example usage with your uploaded file
image_path = "/Users/viveksharma/Desktop/programming/scanner/label1.png"
text = extract_text_from_image(image_path)
ingredient_list = clean_and_split_ingredients(text)
final_results = match_ingredients(ingredient_list)

# Display results
for result in final_results:
    print(result)

{'input': 'aqua', 'match': 'aqua', 'risk': 'Safe', 'desc': 'Water, used as a solvent'}
{'input': 'cyclopentasiloxane', 'match': None, 'risk': 'Unknown', 'desc': 'Ingredient not found in database.'}
{'input': 'glycerine', 'match': 'glycerin', 'risk': 'Safe', 'desc': 'Humectant that hydrates skin'}
{'input': 'caprylic/', 'match': None, 'risk': 'Unknown', 'desc': 'Ingredient not found in database.'}
{'input': 'capric triglyceride', 'match': None, 'risk': 'Unknown', 'desc': 'Ingredient not found in database.'}
{'input': 'dimethicone', 'match': 'dimethicone', 'risk': 'Low', 'desc': 'Silicone used for smoothing'}
{'input': 'isohexadecane', 'match': None, 'risk': 'Unknown', 'desc': 'Ingredient not found in database.'}
{'input': 'cetyl alcohol', 'match': None, 'risk': 'Unknown', 'desc': 'Ingredient not found in database.'}
{'input': 'glyceryl stearal', 'match': None, 'risk': 'Unknown', 'desc': 'Ingredient not found in database.'}
{'input': 'peg-100 stearate', 'match': None, 'risk': 'Unknown', 